<table>
  <tr>
    <td><div align="left"><font size="30">Image-based Visual Servo</font></div></td>
    <td><img src="https://github.com/petercorke/machinevision-toolbox-python/raw/main/docs/figs/VisionToolboxLogo_NoBackgnd@2x.png?raw=1" width="200"></td>
  </tr>
</table>

(c) Peter Corke 2024

Robotics, Vision & Control: Python, see Chapter 15

## Configuring the Jupyter environment
We need to import some packages to help us with linear algebra (`numpy`), graphics (`matplotlib`), and machine vision (`machinevisiontoolbox`).
If you're running locally you need to have these packages installed.  If you're running on CoLab we have to first install machinevisiontoolbox which is not preinstalled, this will be a bit slow.

In [ ]:
# MVTB_BOOTSTRAP_CELL: installs the package, which takes a short time, so wait for the line "Running ..." below.  Generated by sync_bootstrap.py from _mvtb_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'machinevision-toolbox-python'
IMPORT_NAME = 'machinevisiontoolbox'
GIT_URL: str | None = 'https://github.com/petercorke/machinevision-toolbox-python'
JL_PACKAGES = ['numpy', 'scipy', 'matplotlib', 'pillow', 'typing_extensions', 'opencv-python', 'ansitable', 'mvtb-data', 'tqdm', 'requests', 'ipywidgets', 'spatialmath-python', 'pgraph-python']
JL_NO_DEPS: list[str] = []
JL_PACKAGE_NO_DEPS = True
JL_EXTRAS: dict[str, list[str]] = {}
COLAB_PACKAGES: list[str] = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE, deps=not JL_PACKAGE_NO_DEPS)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed()


In [ ]:
import numpy as np
import math

%matplotlib inline

np.set_printoptions(linewidth=120, formatter={'float': lambda x: f"{x:8.4g}" if abs(x) > 1e-10 else f"{0:8.4g}"})
from machinevisiontoolbox import Image, CentralCamera
from spatialmath import base, SE3

import matplotlib.pyplot as plt

***
We will define the coordinates of the three vertices of an equilateral triangle, as columns of a matrix

In [ ]:
P = np.array([  [-1, 0,            5],
                [ 1, 0,            5],
                [ 0, math.sqrt(3), 5]]).T
P

Instantiate a projective camera object, centred at the origin and viewing parallel to the world frame z-axis

In [ ]:
camera = CentralCamera.Default()

Now we can project the triangle vertex points to the image plane

In [ ]:
p = camera.project_point(P)
p

the result is a matrix with one column for each image plane point for the world point in the corresponding column  (in units of pixels)

In [ ]:
# setup the plot
plt.figure()
ax = plt.gca()
plt.grid(True)
ax.set_aspect('equal')
ax.set_facecolor('yellow')
plt.xlabel('u (pixels)')
plt.ylabel('v (pixels)')
plt.title('Camera image plane')
plt.xlim(0, 1000)
ax.set_ylim(1000, 0)  # inverted y-axis

plt.plot(p[0,:], p[1,:], 'o');


Now, we want the image-plane points to be somewhere else, let's choose the goal image plane coordinates

In [ ]:
pstar = np.array( [
    [100,   100],
    [900,   180],
    [500,   870] ]).T

plt.figure()
ax = plt.gca()
plt.grid(True)
ax.set_aspect('equal')
ax.set_facecolor('yellow')
plt.xlabel('u (pixels)')
plt.ylabel('v (pixels)')
plt.title('Camera image plane')
plt.xlim(0, 1000)
ax.set_ylim(1000, 0)  # inverted y-axis

plt.plot(p[0,:], p[1,:], 'o');
plt.plot(pstar[0,:], pstar[1,:], 'r*', markersize=16);
plt.show()

The error, between where the vertices are, and where we want them is simply

In [ ]:
e = pstar - p
print(e)

we can plot the actual image-plane positions, the desired image-plane positions and the error vectors on the one plot

In [ ]:
plt.clf()
plt.figure()
ax = plt.gca()
plt.grid(True)
ax.set_aspect('equal')
ax.set_facecolor('yellow')
plt.xlabel('u (pixels)')
plt.ylabel('v (pixels)')
plt.title('Camera image plane')
plt.xlim(0, 1000)
ax.set_ylim(1000, 0)  # inverted y-axis

plt.plot(p[0,:], p[1,:], 'o')
plt.plot(pstar[0,:], pstar[1,:], 'r*', markersize=16)

for a,b in zip(p.T, e.T):
    plt.arrow(a[0], a[1], b[0], b[1], head_width=20, zorder=20, length_includes_head=True)

The image-plane velocity will be some small multiple of these error vectors

In [ ]:
pdot_star = 0.01 * e.reshape((-1,1), order='F')  # turn e into a column vector with row-major ordering
pdot_star

Now we can create a Jacobian matrix for the 3 points, this is a vertical stack of the $2 \times 6$ Jacobian matrices, resulting in a $6 \times 6$ matrix. `visjac` does this automatically, given N points, it will return a $2N \times 6$ Jacobian matrix.  We do need to provide an estimate of the distance to the points, here that is set to 3m for all the points.  See `Camera.visjac?` for more details.

In [ ]:
Jp = camera.visjac_p(p, 3)
print(Jp.round(2))

Now we invert the relationship from camera velocity to image plane, to solve for the camera velocity required given the desired image plane velocity.

In [ ]:
vcam = np.linalg.inv(Jp) @ pdot_star
vcam

which is the velocity the camera must have in 3D space in order to move the image to be more like that shown with the stars.  The motion is predominantly in the camera x-direction as well as some rotation about the camera's y-axis.

We define an IBVS visual servo control function, this computes the control given the camera pose (the pose attribute of the camera object) and the desired image plane points.

In [ ]:
vmax = 0.2  # maximum norm of camera velocity
dt = 1   # time step

def IBVS(camera, P, pstar, lam=1, Z=5):
    p = camera.project_point(P)
    e = pstar - p
    pdot_star = lam * e.flatten(order='F')
    Jp = camera.visjac_p(p, Z)
    
    # limit speed of motion
    vcam = np.linalg.inv(Jp) @ pdot_star
    vn = np.linalg.norm(vcam)
    if vn > vmax:
        vcam = vcam / vn * vmax
    
    # update camera pose, we use matrix exponential to convert velocity screw to SE(3) increment
    camera.pose @= SE3.Delta(vcam * dt) 
    
    return vcam, np.linalg.norm(e), p

We need to call this function repeatedly to see how the camera pose evolves over time.  

Sorry, too hard to make cool graphics that work on CoLab :(

In [ ]:
# the camera starts at the origin, with its optical axis parallel to the world z-axis
camera = CentralCamera.Default()
P = np.array([  [-1, 0,            5],
                [ 1, 0,            5],
                [ 0, math.sqrt(3), 5]]).T
pstar = np.array( [
                  [100,   100],
                  [900,   180],
                  [500,   870] ]).T

# Set parameters for pose updating

Z = 5       # use fixed value of Z
lam = 0.2   # scalar gain

pose_history = SE3.Empty()
enorm_history = []
p_history = []
for i in range(50):
    vel, enorm, p = IBVS(camera, P, pstar, lam=lam)
    # print(enorm, camera.pose.strline(), vel.T)

    # save results for later
    pose_history.append(camera.pose)
    enorm_history.append(enorm)
    p_history.append(p)

OK, that wasn't very exciting, let's see what happened.

Firstly, we will see how the points moved on the image plane of the camera (which is moving)

In [ ]:
plt.figure()
plt.plot(pstar[0,:], pstar[1,:], 'r*', markersize=16);
plt.xlabel('X')
plt.ylabel('Y')
plt.grid(True)
for p in p_history:
    plt.plot(p[0, :], p[1, :], 'k.', markersize=6)
plt.plot(pstar[0,:], pstar[1,:], 'r*', markersize=16);

Here's how the camera moved.  

First we plot the camera as a simple graphical icon at each time step.  We see that the camera has moved upward (positive z-axis motion)

In [ ]:
base.plotvol3(3);
for pose in pose_history[::2]:
  camera.plot(pose);

Next we plot the positional components of the camera pose

In [ ]:
plt.plot(pose_history.t)
plt.legend(list("XYZ"))
plt.xlabel('time step')
plt.grid(True)

Finally, here's the error norm, which has converged to zero.  It is monotonic, but not exponential decay.

In [ ]:
plt.plot(enorm_history)
plt.xlabel('time step')
plt.ylabel('$\|e\|$')
plt.grid(True)

You can experiment with this in various ways:
* change the gain
* change the initial position of the camera
* servo to more than 3 points. In this case the Jacobian will be non-square so you will have to use the pseudo inverse
* change the assumed distance of the target